# RSNA Knee Abnormality Detection: DINOv2 + 擬似ラベル (Soft Label) 統合パイプライン

## 概要
- **バックボーン**: `facebook/dinov2-small` (または `facebook/dinov2-base`)
- **アテンション構造**: `SlotHead` (12の病変ターゲットごとの Per-Diagnosis Attention Pooling)
- **教師データ**: 元ラベル付き 58件 (Hard Label 1.0) ＋ XLM-RoBERTa ゼロショット擬似ラベル 4,349件 (`pseudo_label_scores.csv` Soft Label 0.7)
- **前処理 & 正規化**: 物理スケールスライス選定、1st-99th percentile輝度正規化、Left-knee Laterality正規化
- **損失関数**: Soft Label (0~1 確率値) に自然に対応する `BCEWithLogitsLoss`


## 1. ライブラリインポート & ハイパーパラメータ設定

In [ ]:
import os, sys, time, glob, re, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoImageProcessor, Dinov2Model
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# パス判定
DATA_DIR = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection/') if os.path.exists('/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv') else Path('/kaggle/input/rsna-knee-abnormality-detection')
MODEL_NAME = '/kaggle/input/models/metaresearch/dinov2/pytorch/small/1'
N_SLOTS = 5
N_SLICES = 3
IMG_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 5
LR = 1e-4
UNFREEZE_LAST = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]
print(f'Device: {DEVICE}, Model: {MODEL_NAME}, Data Dir: {DATA_DIR}')

## 2. 擬似ラベル統合データの準備

未ラベリング4,349件に対して生成した `pseudo_label_scores.csv` (連続確率値 Soft Label) を統合し、データ量を 58件 → 4,407件 へ拡張します。

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
train_series_df = pd.read_csv(DATA_DIR / 'train_series.csv')
pseudo_scores_path = '/kaggle/input/datasets/udaken10/pseudo-label-scores/pseudo_label_scores.csv'

if pseudo_scores_path.exists():
    pseudo_df = pd.read_csv(pseudo_scores_path)
    print(f'擬似ラベル読み込み完了: {len(pseudo_df)} 件')
    
    # 欠損値を pseudo_label_scores の値で補完 (Soft Label 統合)
    pseudo_map = pseudo_df.set_index('StudyInstanceUID')
    for target in TARGETS:
        missing_mask = train_df[target].isna()
        train_df.loc[missing_mask, target] = train_df.loc[missing_mask, 'StudyInstanceUID'].map(pseudo_map[target])
    
    # 重み付け: 元々の Hard Label (58件) は 1.0、擬似ラベル (Soft Label 4,349件) は 0.7
    train_df['is_pseudo'] = train_df['StudyInstanceUID'].isin(pseudo_df['StudyInstanceUID'])
    train_df['sample_weight'] = np.where(train_df['is_pseudo'], 0.7, 1.0)
else:
    print('⚠️ 擬似ラベルが見つかりません。58件のみで進行します。')
    train_df['sample_weight'] = 1.0

print(f'学習対象データ数: {len(train_df)} 症例')
print(train_df[TARGETS].head(5).round(3))

## 3. DINOv2 バックボーン ＆ SlotHead アテンションモデル定義

In [ ]:
class SlotHead(nn.Module):
    """Per-diagnosis attention over the slot embeddings of one study."""
    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden

    def forward(self, x, mask):
        # x: [B, S, dim], mask: [B, S]
        h = self.proj(x) + self.slot_emb
        att = torch.einsum('bsh,oh->bos', h, self.query) / (self.hidden ** 0.5)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum('bos,bsh->boh', att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias

class RSNADinoV2Model(nn.Module):
    def __init__(self, model_name=MODEL_NAME, n_slots=N_SLOTS, num_classes=len(TARGETS), unfreeze_last=UNFREEZE_LAST):
        super().__init__()
        self.backbone = Dinov2Model.from_pretrained(model_name)
        # バックボーンのパラメータ固定/解凍制御
        for param in self.backbone.parameters():
            param.requires_grad = False
        if unfreeze_last > 0 and hasattr(self.backbone, 'encoder'):
            for layer in self.backbone.encoder.layer[-unfreeze_last:]:
                for param in layer.parameters():
                    param.requires_grad = True
        
        dim = self.backbone.config.hidden_size * 2  # CLS token + Spatial Token Mean
        self.head = SlotHead(dim, n_slots, num_classes)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask):
        # imgs: [B, S, C, H, W]
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float() / 255.0
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        feat = torch.cat([out[:, 0], out[:, 1:].mean(1)], dim=1).reshape(B, S, -1)
        return self.head(feat, mask)

print('RSNADinoV2Model & SlotHead 定義完了')

## 4. Soft Label 対応損失関数 & AUC 評価

In [ ]:
def weighted_bce_loss(logits, targets, weights=None):
    """Soft Label (0~1) に対応した BCEWithLogitsLoss"""
    loss_per_element = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    if weights is not None:
        loss_per_element = loss_per_element * weights.unsqueeze(1)
    return loss_per_element.mean()

def compute_macro_auc(y_true, y_pred_probs):
    """12病変ターゲットに対する Macro AUC-ROC の計算"""
    aucs = []
    for i, target in enumerate(TARGETS):
        # 二値化したラベルに対して AUC を計算
        t = (y_true[:, i] >= 0.5).astype(int)
        p = y_pred_probs[:, i]
        if len(np.unique(t)) > 1:
            auc = roc_auc_score(t, p)
            aucs.append(auc)
    return np.mean(aucs) if len(aucs) > 0 else 0.5

print('損失関数 ＆ Macro-AUC 評価関数完了')

## 5. 次のステップ・まとめ

1. `pseudo_label_scores.csv` の 4,349 件が滑らかな Soft Label として統合されました。
2. `RSNADinoV2Model` が DINOv2 の表現能力と `SlotHead` の多視点アテンションを接続しました。
3. `weighted_bce_loss` が確信度重み付きで学習を導きます。